In [ ]:
import pandas as pd
from ydata_profiling import ProfileReport

# Temizlenmiş veriyi okuyoruz
df_oyunlar = pd.read_csv("../../data/processed/oyunlar_temiz.csv")

# 1. AGRESİF KALKAN: Sadece oyun adını değil, binlerce farklı metin içeren geliştiriciyi de çıkarıyoruz.
df_eda = df_oyunlar.drop(columns=['oyun_adi', 'gelistirici'], errors='ignore')

# 2. HIZLI MOD: minimal=True ile devam
profil_raporu = ProfileReport(df_eda, title="Oyun Verileri EDA Raporu", minimal=True)

# 3. KİLİTLENME ÇÖZÜMÜ: to_widgets() komutunu TAMAMEN siliyoruz! 
# Notebook içinde göstermeye çalışıp VS Code'u boğmayacağız.

# Doğrudan reports klasörüne kaydediyoruz
rapor_yolu = "../../reports/oyunlar_eda_raporu.html"
profil_raporu.to_file(rapor_yolu)

print(f"✅ Rapor oluşturma işlemi bitti!")
print(f"Lütfen sol taraftaki dosya yöneticisinden '{rapor_yolu}' dosyasını bulup normal bir tarayıcıda (Chrome, Edge vb.) açın.")

In [1]:
import pandas as pd
import plotly.express as px
from IPython.display import display

def plotly_otomatik_eda(df):
    print("="*60)
    print("📊 VERİ SETİ GENEL BAKIŞ (SUMMARY)")
    print("="*60)
    print(f"Toplam Satır: {df.shape[0]}")
    print(f"Toplam Sütun: {df.shape[1]}")
    
    # Eksik Veri Analizi Tablosu
    eksik_df = pd.DataFrame({
        'Eksik Değer Sayısı': df.isnull().sum(),
        'Eksik Oranı (%)': (df.isnull().sum() / len(df) * 100).round(2)
    })
    print("\nEksik Veri (NaN) Analizi:")
    display(eksik_df[eksik_df['Eksik Değer Sayısı'] > 0].sort_values(by='Eksik Oranı (%)', ascending=False))
    
    print("\n" + "="*60)
    print("📈 SÜTUN BAZLI GÖRSEL DAĞILIMLAR")
    print("="*60)

    for col in df.columns:
        s_valid = df[col].dropna()
        if s_valid.empty:
            continue

        nunique = s_valid.nunique()

        # 1. KATEGORİK VEYA METİN VERİLER (Platform, Cihaz, Kategori, Geliştirici)
        if df[col].dtype == 'object' or str(df[col].dtype) == 'category' or nunique <= 20:
            if nunique > 15:
                # Kilitlenmeyi önlemek için sadece en çok geçen 15 değeri al
                title = f"'{col}' Dağılımı (En Çok Geçen İlk 15) - Toplam {nunique} Farklı Değer"
                val_counts = s_valid.value_counts().head(15).reset_index()
            else:
                title = f"'{col}' Dağılımı - Toplam {nunique} Farklı Değer"
                val_counts = s_valid.value_counts().reset_index()

            val_counts.columns = [col, 'Frekans']

            # Bar Grafiği Çizimi
            fig = px.bar(val_counts, x=col, y='Frekans', title=title, 
                         text='Frekans', color='Frekans', color_continuous_scale='Viridis')
            fig.update_traces(textposition='outside')
            fig.update_layout(xaxis_tickangle=-45, height=450)
            fig.show()

        # 2. SAYISAL VERİLER (Fiyat, İndirilme Sayısı)
        elif pd.api.types.is_numeric_dtype(df[col]):
            title = f"'{col}' Sayısal Dağılımı ve Aykırı Değerler"
            # Histogram ve üstüne Kutu Grafiği (Boxplot) eklenmiş harika görünüm
            fig = px.histogram(df, x=col, title=title, marginal="box", 
                               nbins=50, color_discrete_sequence=['indianred'])
            fig.update_layout(height=450)
            fig.show()

# ---------------------------------------------------------
# Veriyi Çekme ve Fonksiyonu Çalıştırma İşlemi
# ---------------------------------------------------------

# 1. Veriyi okuyoruz
dosya_yolu = "../../data/processed/oyunlar_temiz.csv"
df_oyunlar = pd.read_csv(dosya_yolu)

# 2. Orijinal oyun adı sütununu grafikleri boğmaması için EDA öncesi geçici olarak siliyoruz
df_eda_icin = df_oyunlar.drop(columns=['oyun_adi'], errors='ignore')

# 3. Kendi yazdığımız fonksiyonu ateşliyoruz!
plotly_otomatik_eda(df_eda_icin)

📊 VERİ SETİ GENEL BAKIŞ (SUMMARY)
Toplam Satır: 21839
Toplam Sütun: 7

Eksik Veri (NaN) Analizi:


,Eksik Değer Sayısı,Eksik Oranı (%)
indirilme_sayisi,21839,100.00
gelistirici,8557,39.18
fiyat,1853,8.48
yas_siniri,1065,4.88
kategori,292,1.34



📈 SÜTUN BAZLI GÖRSEL DAĞILIMLAR
